<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix C · Risk-Neutral Valuation

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_c_risk_neutral_valuation.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

After working through this notebook, you will be able to:

- distinguish the real-world measure used for forecasting and risk analysis from the risk-neutral measure used for pricing;
- define the money-market account as one currency unit rolled at the short rate and explain why discounted traded prices are martingales under the pricing measure;
- state the pricing formula as the current numeraire times the discounted expectation of the payoff, and its constant-rate special case as the exponential discount factor times the expectation;
- connect zero-coupon bond prices to expected discount factors; and
- explain why credit spreads imply risk-neutral default compensation rather than physical default frequencies.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

The appendix is the pricing backbone of the term-structure models in Chapter 12, the rate-option formulas in Chapter 14, and the reduced-form credit models in Chapter 17. The companion figure script `code/figures/appx_c_risk_neutral_pricing_weights.py` contrasts real-world probabilities with risk-neutral pricing weights in a two-state example. This notebook complements the derivation with a Monte Carlo experiment under the risk-neutral measure: a binary payoff that pays one unit when the terminal value finishes above the strike is priced as a discounted expectation, and the estimate is checked against the closed-form value, the discounted normal probability N(d2).

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_c_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

Answer these before using the appendix result in a chapter calculation:

- Under which measure is the expectation taken, and is the numeraire the money-market account or another asset?
- Is the short rate deterministic, so that discounting factors out, or stochastic, so that the discount factor stays inside the expectation?
- Are the probabilities in your calculation pricing weights or forecasts, and would the real-world measure give a different value?
- What are the payoff's maturity, strike, settlement, and units, and is the payoff contingent on default or another credit event?
- How large is the Monte Carlo standard error, and is the estimate within tolerance of the analytic benchmark?

## Mini-lab: Price a binary payoff by discounted risk-neutral expectation

This lab isolates the pricing formula: value equals the exponential discount factor times the expectation under the risk-neutral measure Q. The underlying starts at 100, the strike is 105, the constant continuously compounded short rate is 4 percent per year, and the volatility is 20 percent per year over a one-year horizon, all under Q with the money-market account as numeraire. The binary option pays one unit if the terminal value exceeds the strike, so its value equals the discounted Q-probability of finishing in the money.

In [ ]:
import math
import numpy as np
import pandas as pd

rng = np.random.default_rng(100)
s0, strike, r, sigma, T = 100.0, 105.0, 0.04, 0.20, 1.0
paths = 50_000
z = rng.standard_normal(paths)  # draws under the pricing measure
s_t = s0 * np.exp((r - 0.5 * sigma ** 2) * T
                  + sigma * np.sqrt(T) * z)  # terminal values
payoff = (s_t > strike).astype(float)  # binary payoff: 1 or 0
price = np.exp(-r * T) * payoff.mean()  # discounted expectation
d2 = (np.log(s0 / strike) + (r - 0.5 * sigma ** 2) * T) / (
    sigma * np.sqrt(T))
analytic = np.exp(-r * T) * 0.5 * (1 + math.erf(d2 / np.sqrt(2)))
pd.Series({"monte_carlo_price": price, "analytic_price": analytic})

### What to observe

With seed 100 and 50,000 paths, the Monte Carlo value is about 0.428 against the analytic discounted N(d2) of about 0.425, within sampling error. For comparison, the one-period binomial example in the appendix gives a risk-neutral probability of 0.60 and a call value of 5.88.

## Exercises

1. **Reproduce the binomial example.** With a spot of 100, up to 110 or down to 90, and a gross risk-free return of 1.02, verify the risk-neutral probability of 0.60 and the call value of 5.88. *Hint:* Choose the probability so that the discounted underlying is a martingale.
2. **Contrast the real-world measure.** Recompute the binomial value with an up probability of 0.70 and explain why the result, 6.86, is not a price. *Hint:* The physical probability does not reproduce the traded underlying price.
3. **Price a zero-coupon bond.** Under a constant rate of 4 percent, price a default-free bond paying 1 at a two-year maturity and explain why the expectation drops out. *Hint:* The discounted payoff is known, so the value is the exponential of minus 0.08.
4. **Size the Monte Carlo error.** Rerun the binary pricing with 5,000 and 500,000 paths and compare the spread of the estimates around the analytic value. *Hint:* The standard error of a binary estimate scales with the reciprocal of the square root of the number of paths.
5. **Move to credit.** Explain why a credit spread of 200 basis points does not mean a 2 percent annual physical default probability. *Hint:* The risk-neutral intensity also compensates for risk aversion, liquidity, and market technicals.

For more detail, see Appendix C of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">